In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, GlobalMaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle


final_train_df = pd.read_csv("/content/final_toxic_comments_train.csv")

final_train_df["threat_emoji"] = final_train_df["threat"]
final_train_df.drop(columns=["Unnamed: 0", "threat.1"], inplace=True)

final_train_df["general_toxic"] = (
    final_train_df["toxic"] |
    final_train_df["insult"] |
    final_train_df["obscene"]
)

final_train_df["threat_harm"] = (
    final_train_df["threat"] |
    final_train_df["severe_toxic"]
)

final_train_df["identity_hate"] = final_train_df["identity_hate"]

labels = ["general_toxic", "threat_harm", "identity_hate"]
y = final_train_df[labels].values

max_words = 20000
max_len = 150

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(final_train_df["comment_text"])

X = tokenizer.texts_to_sequences(final_train_df["comment_text"])
X = pad_sequences(X, maxlen=max_len, padding="post")


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 5. MODEL BUILDING
# =========================
model = Sequential()

# Embedding layer
model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))

# LSTM layer
model.add(LSTM(64, return_sequences=True))
model.add(GlobalMaxPooling1D())

# Dense layers
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.5))

# Output layer (multi-label → sigmoid)
model.add(Dense(3, activation="sigmoid"))

# =========================
# 6. COMPILE MODEL
# =========================
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=[
        tf.keras.metrics.Precision(),
        tf.keras.metrics.Recall()
    ]
)

# =========================
# 7. CALLBACKS (optional but good)
# =========================
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)
checkpoint=ModelCheckpoint("my_toxic_classifier.keras")

# =========================
# 8. TRAIN MODEL
# =========================
weights = np.ones(len(y_train))

# increase weight for rare labels
weights += (y_train[:, 1] * 2)   # threat_harm
weights += (y_train[:, 2] * 3)   # identity_hate



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [2]:

history = model.fit(
    X_train,
    y_train,
    sample_weight=weights,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop,checkpoint]
)






# =========================
# 11. SAVE MODEL + TOKENIZER
# =========================

Epoch 1/10
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 646s 161ms/step - loss: 0.1250 - precision: 0.7456 - recall: 0.5591 - val_loss: 0.0590 - val_precision: 0.8365 - val_recall: 0.6260
Epoch 2/10
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 578s 145ms/step - loss: 0.0869 - precision: 0.7935 - recall: 0.6897 - val_loss: 0.0565 - val_precision: 0.8010 - val_recall: 0.6778
Epoch 3/10
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 621s 145ms/step - loss: 0.0706 - precision: 0.7922 - recall: 0.7660 - val_loss: 0.0574 - val_precision: 0.7270 - val_recall: 0.7335
Epoch 4/10
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 629s 158ms/step - loss: 0.0597 - precision: 0.8103 - recall: 0.8196 - val_loss: 0.0628 - val_precision: 0.7363 - val_recall: 0.7348
998/998 ━━━━━━━━━━━━━━━━━━━━ 33s 32ms/step


ValueError: Classification metrics can't handle a mix of multilabel-indicator and continuous-multioutput targets

In [17]:

# =========================
# 9. PREDICTION
# =========================
y_pred = model.predict(X_test)
thresholds = [0.5, 0.4,0.4]

y_pred_bin = np.zeros_like(y_pred)

for i in range(3):
    y_pred_bin[:, i] = (y_pred[:, i] > thresholds[i]).astype(int)


998/998 ━━━━━━━━━━━━━━━━━━━━ 37s 37ms/step


In [18]:
# =========================
# 10. EVALUATION
# =========================
print(classification_report(y_test, y_pred_bin, target_names=labels))

               precision    recall  f1-score   support

general_toxic       0.87      0.73      0.80      3233
  threat_harm       0.41      0.66      0.50       387
identity_hate       0.48      0.44      0.46       294

    micro avg       0.76      0.70      0.73      3914
    macro avg       0.59      0.61      0.59      3914
 weighted avg       0.80      0.70      0.74      3914
  samples avg       0.07      0.07      0.07      3914



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [5]:
model.save("toxicity_model.keras")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("config.pkl", "wb") as f:
    pickle.dump({
        "max_len": max_len,
        "labels": labels
    }, f)

In [19]:
thresholds = [0.5, 0.45, 0.5]

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, Dense, Dropout,
    GlobalMaxPooling1D, Concatenate
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("/content/final_toxic_comments_train.csv")

df.drop(columns=["Unnamed: 0", "threat.1"], inplace=True)

# =========================
# 2. LABELS
# =========================
df["general_toxic"] = df["toxic"] | df["insult"] | df["obscene"]

df["threat_harm"] = df["threat"] | df["severe_toxic"]

labels = ["general_toxic", "threat_harm", "identity_hate"]
y = df[labels].values

# =========================
# 3. TEXT PROCESSING
# =========================
max_words = 20000
max_len = 150

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df["comment_text"])

X_text = tokenizer.texts_to_sequences(df["comment_text"])
X_text = pad_sequences(X_text, maxlen=max_len, padding="post")

# =========================
# 4. ENGINEERED FEATURES
# =========================
X_feat = df.drop(columns=["comment_text"] + labels)

X_feat = X_feat.astype(np.float32).values

# =========================
# 5. TRAIN TEST SPLIT (TWO INPUTS)
# =========================
X_text_train, X_text_test, X_feat_train, X_feat_test, y_train, y_test = train_test_split(
    X_text, X_feat, y,
    test_size=0.2,
    random_state=42
)

# =========================
# 6. MODEL INPUTS
# =========================
text_input = Input(shape=(max_len,), name="text_input")
feat_input = Input(shape=(X_feat.shape[1],), name="feat_input")

# =========================
# 7. TEXT BRANCH
# =========================
x = Embedding(input_dim=max_words, output_dim=128)(text_input)
x = LSTM(64, return_sequences=True)(x)
x = GlobalMaxPooling1D()(x)

# =========================
# 8. FEATURE BRANCH
# =========================
y_feat = Dense(64, activation="relu")(feat_input)
y_feat = Dropout(0.3)(y_feat)

# =========================
# 9. MERGE
# =========================
combined = Concatenate()([x, y_feat])

z = Dense(64, activation="relu")(combined)
z = Dropout(0.5)(z)

output = Dense(3, activation="sigmoid")(z)

# =========================
# 10. MODEL
# =========================
model1 = Model(inputs=[text_input, feat_input], outputs=output)

model1.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.Precision(),
        tf.keras.metrics.Recall()
    ]
)

model1.summary()

# =========================
# 11. TRAIN
# =========================
# =========================
# 8. TRAIN MODEL
# =========================
weights = np.ones(len(y_train))

# increase weight for rare labels
weights += (y_train[:, 1] * 2)   # threat_harm
weights += (y_train[:, 2] * 3)   # identity_hate
history = model1.fit(
    [X_text_train, X_feat_train],
    y_train,
    validation_data=([X_text_test, X_feat_test], y_test),
    epochs=5,
    sample_weight=weights,
    batch_size=32
)

# =========================
# 12. PREDICT
# =========================
y_pred = model1.predict([X_text_test, X_feat_test])
y_pred = (y_pred > 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=labels))

# =========================
# 13. SAVE
# =========================
model1.save("toxicity_model1.keras")

with open("tokenizer1.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("config1.pkl", "wb") as f:
    pickle.dump({
        "max_len": max_len,
        "labels": labels
    }, f)